In [3]:
from pathlib import Path
import pandas as pd
import os

PROJECT_ROOT = Path.cwd().resolve().parents[0]

RAW_ONET = PROJECT_ROOT / "Data" / "raw" / "onet" / "db_29_0_text"
print("O*NET path:", RAW_ONET)
print("Exists:", RAW_ONET.exists())

O*NET path: E:\AI_Prsonalised_path\Data\raw\onet\db_29_0_text
Exists: True


In [4]:
onet_files = sorted(RAW_ONET.glob("*.txt"))

print(f"Total O*NET files: {len(onet_files)}\n")

for f in onet_files:
    print(f"{f.name:50} {f.stat().st_size / 1024 / 1024:.2f} MB")

Total O*NET files: 41

Abilities to Work Activities.txt                   0.03 MB
Abilities to Work Context.txt                      0.01 MB
Abilities.txt                                      8.02 MB
Alternate Titles.txt                               2.27 MB
Basic Interests to RIASEC.txt                      0.00 MB
Content Model Reference.txt                        0.09 MB
DWA Reference.txt                                  0.18 MB
Education, Training, and Experience Categories.txt 0.00 MB
Education, Training, and Experience.txt            3.37 MB
Emerging Tasks.txt                                 0.03 MB
Interests Illustrative Activities.txt              0.01 MB
Interests Illustrative Occupations.txt             0.01 MB
Interests.txt                                      0.55 MB
IWA Reference.txt                                  0.02 MB
Job Zone Reference.txt                             0.00 MB
Job Zones.txt                                      0.03 MB
Knowledge.txt                    

In [5]:
def read_onet_file(filename):
    path = RAW_ONET / filename
    
    try:
        df = pd.read_csv(
            path,
            sep="\t",
            encoding="utf-8",
            low_memory=False
        )
    except UnicodeDecodeError:
        df = pd.read_csv(
            path,
            sep="\t",
            encoding="latin1",
            low_memory=False
        )
    
    return df

In [6]:
files_to_load = {
    "occupations": "Occupation Data.txt",
    "skills": "Skills.txt",
    "knowledge": "Knowledge.txt",
    "abilities": "Abilities.txt",
    "work_activities": "Work Activities.txt",
    "tasks": "Task Statements.txt",
    "task_ratings": "Task Ratings.txt",
    "technology": "Technology Skills.txt",
    "tools": "Tools Used.txt",
    "work_styles": "Work Styles.txt",
    "work_values": "Work Values.txt",
    "job_zones": "Job Zones.txt",
    "education": "Education, Training, and Experience.txt",
    "related_occupations": "Related Occupations.txt",
    "emerging_tasks": "Emerging Tasks.txt",
    "alternate_titles": "Alternate Titles.txt",
    "reported_titles": "Sample of Reported Titles.txt",
    "skills_activities": "Skills to Work Activities.txt",
    "skills_context": "Skills to Work Context.txt",
    "abilities_activities": "Abilities to Work Activities.txt",
    "abilities_context": "Abilities to Work Context.txt",
}

In [7]:
onet_data = {}

for key, filename in files_to_load.items():
    print(f"Loading: {filename}")
    
    try:
        df = read_onet_file(filename)
        onet_data[key] = df
        
        print(f"  Shape: {df.shape}")
        print(f"  Columns: {df.columns.tolist()}")
        print()
        
    except Exception as e:
        print(f"  ERROR: {e}")
        print()

Loading: Occupation Data.txt
  Shape: (1016, 3)
  Columns: ['O*NET-SOC Code', 'Title', 'Description']

Loading: Skills.txt
  Shape: (61530, 13)
  Columns: ['O*NET-SOC Code', 'Element ID', 'Element Name', 'Scale ID', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date', 'Domain Source']

Loading: Knowledge.txt
  Shape: (58014, 13)
  Columns: ['O*NET-SOC Code', 'Element ID', 'Element Name', 'Scale ID', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date', 'Domain Source']

Loading: Abilities.txt
  Shape: (91416, 13)
  Columns: ['O*NET-SOC Code', 'Element ID', 'Element Name', 'Scale ID', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date', 'Domain Source']

Loading: Work Activities.txt
  Shape: (72078, 13)
  Columns: ['O*NET-SOC Code', 'Element ID', 'Element Name', 'Scale ID', 'Data Value', 'N', 

In [8]:
summary = []

for name, df in onet_data.items():
    summary.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
    })

onet_summary = pd.DataFrame(summary)

print(onet_summary.to_string(index=False))

             dataset   rows  columns  missing_cells  duplicate_rows
         occupations   1016        3              0               0
              skills  61530       13          30765               0
           knowledge  58014       13          87513               0
           abilities  91416       13          45708               0
     work_activities  72078       13         107087               0
               tasks  18796        7           2796               0
        task_ratings 158751       12         225057               0
          technology  32470        6              0               0
               tools  41650        4              0               0
         work_styles  14064       12          13860               0
         work_values   7866        7              0               0
           job_zones    923        4              0               0
           education  36209       13          54959               0
 related_occupations  18460        4            

In [9]:
for name, df in onet_data.items():
    print("\n" + "=" * 100)
    print(name.upper())
    print("=" * 100)
    
    for col in df.columns:
        col_lower = col.lower()
        
        if any(x in col_lower for x in [
            "soc",
            "occupation",
            "element",
            "skill",
            "ability",
            "knowledge",
            "task",
            "activity",
            "technology",
            "tool",
            "zone",
            "scale",
            "title",
            "id"
        ]):
            print(col)


OCCUPATIONS
O*NET-SOC Code
Title

SKILLS
O*NET-SOC Code
Element ID
Element Name
Scale ID

KNOWLEDGE
O*NET-SOC Code
Element ID
Element Name
Scale ID

ABILITIES
O*NET-SOC Code
Element ID
Element Name
Scale ID

WORK_ACTIVITIES
O*NET-SOC Code
Element ID
Element Name
Scale ID

TASKS
O*NET-SOC Code
Task ID
Task
Task Type

TASK_RATINGS
O*NET-SOC Code
Task ID
Scale ID

TECHNOLOGY
O*NET-SOC Code
Commodity Title
Hot Technology

TOOLS
O*NET-SOC Code
Commodity Title

WORK_STYLES
O*NET-SOC Code
Element ID
Element Name
Scale ID

WORK_VALUES
O*NET-SOC Code
Element ID
Element Name
Scale ID

JOB_ZONES
O*NET-SOC Code
Job Zone

EDUCATION
O*NET-SOC Code
Element ID
Element Name
Scale ID

RELATED_OCCUPATIONS
O*NET-SOC Code
Related O*NET-SOC Code

EMERGING_TASKS
O*NET-SOC Code
Task
Original Task ID
Original Task

ALTERNATE_TITLES
O*NET-SOC Code
Alternate Title
Short Title

REPORTED_TITLES
O*NET-SOC Code
Reported Job Title

SKILLS_ACTIVITIES
Skills Element ID
Skills Element Name
Work Activities Element ID
Wo

In [10]:
for name, df in onet_data.items():
    print("\n" + "=" * 100)
    print(name.upper())
    print("=" * 100)
    print(df.head(3).to_string(index=False))


OCCUPATIONS
O*NET-SOC Code                           Title                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        Description
    11-1011.00                Chief Executives                                                                                                                                                                                               Determine and formulate policies and provide overall direction of companies or private and public sector organizations within guidelines set up by a board of directors or s

In [11]:
for name, df in onet_data.items():
    occupation_cols = [
        c for c in df.columns
        if "occupation" in c.lower() or "soc" in c.lower()
    ]
    
    if occupation_cols:
        print(f"{name:25} -> {occupation_cols}")

occupations               -> ['O*NET-SOC Code']
skills                    -> ['O*NET-SOC Code']
knowledge                 -> ['O*NET-SOC Code']
abilities                 -> ['O*NET-SOC Code']
work_activities           -> ['O*NET-SOC Code']
tasks                     -> ['O*NET-SOC Code']
task_ratings              -> ['O*NET-SOC Code']
technology                -> ['O*NET-SOC Code']
tools                     -> ['O*NET-SOC Code']
work_styles               -> ['O*NET-SOC Code']
work_values               -> ['O*NET-SOC Code']
job_zones                 -> ['O*NET-SOC Code']
education                 -> ['O*NET-SOC Code']
related_occupations       -> ['O*NET-SOC Code', 'Related O*NET-SOC Code']
emerging_tasks            -> ['O*NET-SOC Code']
alternate_titles          -> ['O*NET-SOC Code']
reported_titles           -> ['O*NET-SOC Code']


In [14]:
for name, df in onet_data.items():
    mask = pd.Series(False, index=df.index)
    
    for col in df.columns:
        if df[col].dtype == "object":
            mask |= df[col].astype(str).str.contains(
                "Data Scientists",
                case=False,
                na=False
            )
    
    result = df[mask]
    
    if len(result):
        print("\n" + "=" * 100)
        print(name.upper())
        print("=" * 100)
        print(result.head(10).to_string(index=False))

In [13]:
for name, df in onet_data.items():
    print("\n" + "=" * 100)
    print(name.upper())
    print("=" * 100)
    
    for col in df.columns:
        if (
            "id" in col.lower()
            or "code" in col.lower()
            or "soc" in col.lower()
        ):
            print(
                f"{col:35} "
                f"unique={df[col].nunique():8} "
                f"missing={df[col].isna().sum():8}"
            )


OCCUPATIONS
O*NET-SOC Code                      unique=    1016 missing=       0

SKILLS
O*NET-SOC Code                      unique=     879 missing=       0
Element ID                          unique=      35 missing=       0
Scale ID                            unique=       2 missing=       0

KNOWLEDGE
O*NET-SOC Code                      unique=     879 missing=       0
Element ID                          unique=      33 missing=       0
Scale ID                            unique=       2 missing=       0

ABILITIES
O*NET-SOC Code                      unique=     879 missing=       0
Element ID                          unique=      52 missing=       0
Scale ID                            unique=       2 missing=       0

WORK_ACTIVITIES
O*NET-SOC Code                      unique=     879 missing=       0
Element ID                          unique=      41 missing=       0
Scale ID                            unique=       2 missing=       0

TASKS
O*NET-SOC Code                      